## Synthetic Data Generation

In [ ]:
from datalink.simulator import generate_synthetic_log_returns
from utils.datastructs import SVMParameters
from scipy import stats
import numpy as np
from estimator import Hyperparameters, FastBayesianSVMEstimator

params = SVMParameters(beta_0=0.2, beta_1=0.07, beta_2=-0.18, mu=0.1, phi=0.98, sigma_eta=0.1)

data = generate_synthetic_log_returns(params, n=3000)

# 1. Define the SMN log-density function (e.g., Student-t distribution)
def student_t_logpdf(y: float, mu: np.ndarray, sigma: np.ndarray, nu: float) -> np.ndarray:
    """
    Log-density of the Student-t distribution parameterized by location, scale, and df.
    """
    # Using scipy stats, passing arrays for vectorized operations over the grid
    return stats.t.logpdf(y, df=nu, loc=mu, scale=sigma)

# 2. Generate/Load Mock Data (1000 days of random returns for demonstration)
np.random.seed(42)
mock_returns = data['Log_Return'].to_numpy()

# 3. Configure Hyperparameters
config = Hyperparameters(
    m=50,               # Lowered to 50 for speed in MWE, paper uses 100-200
    b_limit=2.5,        # Grid boundaries
    is_samples=500      # Samples for IS
)

# 4. Instantiate and run the estimator
estimator = FastBayesianSVMEstimator(
    data=mock_returns,
    smn_logpdf=student_t_logpdf,
    hyperparams=config
)

# 5. Extract Results
result = estimator.estimate()

print("\n--- ESTIMATION RESULTS ---")
print(f"Optimization Success: {result.success}")
print("\nPosterior Means (Constrained Space):")
for param, value in result.posterior_mean_con.items():
    print(f"{param:>10}: {value:.4f}")

2026-08-11 22:28:57,093 [INFO] SVMEstimator: Starting numerical maximization (L-BFGS-B)...
d:\Source\svm-using-hmm\estimator.py:188: OptimizeWarning: Unknown solver options: disp
  opt_res = minimize(
2026-08-11 22:30:06,850 [INFO] SVMEstimator: MAP optimization complete. Unconstrained Mode: [ 0.156  0.158 -0.139  0.192  6.772 -2.654 -0.858]
2026-08-11 22:30:06,854 [INFO] SVMEstimator: Drawing 500 samples for Importance Sampling inference...
2026-08-11 22:33:32,809 [INFO] SVMEstimator: Importance Sampling complete.



--- ESTIMATION RESULTS ---
Optimization Success: True

Posterior Means (Constrained Space):
     beta0: 0.1552
     beta1: 0.0845
     beta2: -0.1139
        mu: 1.4215
       phi: 0.9977
 sigma_eta: 0.0809
        nu: 8.5604
